In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import defaultdict
from scipy.stats import gmean

In [ ]:
aquaculture_chem = pd.read_csv('./Data/MAIN_chemical_structure_data.tsv', sep='\t')
aquaculture_chem = aquaculture_chem.fillna('')
print(aquaculture_chem.shape)
aquaculture_chem.head()

In [ ]:
aquaculture_chem['casrn'] = aquaculture_chem['casrn'].str.replace('CAS:', '')
aquaculture_chem = aquaculture_chem[aquaculture_chem['casrn']!='']
chemdict = dict(zip(aquaculture_chem['casrn'], aquaculture_chem['id']))
len(chemdict)

In [ ]:
def analyze_chemical_predator_paths(G, node_table):
    """
    This function analyzes chemical -> prey -> predator paths where shortest path length >= 2
    (i.e., at least one intermediate prey node exists).

    Sensitive edges (Type == 'Sensitive') are removed before analysis.
    A prey node left with in-degree 0 after that removal is also dropped
    (cascade), since it can no longer receive a chemical transfer.

    Only chemicals and predators connected via such paths are counted.
    Top prey species by path frequency are reported.
    """

    G_filtered = nx.DiGraph()
    G_filtered.add_nodes_from(G.nodes(data=True))
    for u, v, data in G.edges(data=True):
        if data.get('Type') != 'Sensitive':
            G_filtered.add_edge(u, v, **data)

    sensitive_removed = G.number_of_edges() - G_filtered.number_of_edges()

    prey_set_full = set(node_table[node_table['node_type'].isin(['prey', 'both'])]['node'])
    cascade_removed = []
    while True:
        to_drop = [n for n in prey_set_full if n in G_filtered and G_filtered.in_degree(n) == 0]
        if not to_drop:
            break
        for n in to_drop:
            cascade_removed.append(n)
            G_filtered.remove_node(n)
            prey_set_full.discard(n)

    isolates = list(nx.isolates(G_filtered))

    print(" Filtering stats ")
    print(f"  Sensitive edges removed : {sensitive_removed}")
    print(f"  Prey nodes cascaded out : {len(cascade_removed)}{f'  {sorted(cascade_removed)}' if cascade_removed else ''}")
    print(f"  Degree-0 isolates left  : {len(isolates)}{f'  {sorted(isolates)}' if isolates else ''}")
    print(f"  Nodes after filtering   : {G_filtered.number_of_nodes()}  (was {G.number_of_nodes()})")
    print(f"  Edges after filtering   : {G_filtered.number_of_edges()}  (was {G.number_of_edges()})")

    # - Node type sets -
    chemicals = set(node_table[node_table['node_type'] == 'chemical']['node'])
    prey      = set(node_table[node_table['node_type'].isin(['prey', 'both'])]['node'])
    predators = set(node_table[node_table['node_type'].isin(['predator', 'both'])]['node'])

    # - Filter to pairs with shortest path length >= 2 -
    chemical_to_predators = defaultdict(set)
    pred_from_chem        = defaultdict(set)
    prey_path_counts      = defaultdict(int)

    for chem in chemicals:
        for pred in predators:
            try:
                path = nx.shortest_path(G_filtered, chem, pred)
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue

            if len(path) - 1 < 2:
                continue

            chemical_to_predators[chem].add(pred)
            pred_from_chem[pred].add(chem)

            for node in path[1:-1]:
                if node in prey:
                    prey_path_counts[node] += 1

    # - Chemical ranking -
    chemical_impact = {c: len(preds) for c, preds in chemical_to_predators.items()}
    summary_df = (
        pd.DataFrame([
            {'chemical': c, 'num_predators_affected': n}
            for c, n in chemical_impact.items()
        ])
        .sort_values('num_predators_affected', ascending=False)
        .reset_index(drop=True)
    )
    print("\n Chemicals ranked by predators affected (shortest path >= 2) ")
    print(summary_df.head(10).to_string(index=False))

    # - Predator ranking -
    pred_impact = {p: len(chems) for p, chems in pred_from_chem.items()}
    pred_df = (
        pd.DataFrame([
            {'predator': p, 'num_chemicals_affecting': n}
            for p, n in pred_impact.items()
        ])
        .sort_values('num_chemicals_affecting', ascending=False)
        .reset_index(drop=True)
    )
    print("\n Predators ranked by chemicals affecting them (shortest path >= 2) ")
    print(pred_df.head(10).to_string(index=False))

    if pred_impact:
        max_pred = max(pred_impact, key=pred_impact.get)
        min_pred = min(pred_impact, key=pred_impact.get)
        print(f"\nMost impacted predator : {max_pred} ({pred_impact[max_pred]} chemicals)")
        print(f"Least impacted predator: {min_pred} ({pred_impact[min_pred]} chemicals)")

    # - Top prey by appearance in qualifying shortest paths 
    prey_df = (
        pd.DataFrame([
            {'prey': node, 'path_count': cnt}
            for node, cnt in prey_path_counts.items()
        ])
        .sort_values('path_count', ascending=False)
        .reset_index(drop=True)
    )
    print("\n Top 10 prey species in qualifying shortest paths ")
    print(prey_df.head(10).to_string(index=False))

    return {
        'chemical_to_predators': dict(chemical_to_predators),
        'pred_from_chem':        dict(pred_from_chem),
        'chemical_ranking':      summary_df,
        'predator_ranking':      pred_df,
        'prey_path_counts':      prey_df,
    }

# Acute toxicity SSD

### Acute HC05 values

In [ ]:
acute_ssd = pd.read_csv('./external_data/acute_averaged_hc05_all_chemicals.tsv', sep='\t')
print(acute_ssd.shape)
acute_ssd.head()

In [ ]:
print(len(set(acute_ssd['Chemical'])))
print(len(set(acute_ssd['is_bestfit'])))

In [ ]:
acute_ssd['id'] = acute_ssd['cas'].apply(lambda x: chemdict[x] if x in set(chemdict.keys()) else x)
print(acute_ssd.shape)
acute_ssd.head()

In [ ]:
acute_ssd_hc5 = dict(zip(acute_ssd['id'], acute_ssd['est']))
len(acute_ssd_hc5)

### Acute trophic network

In [ ]:
acute_edgelist = pd.read_csv('./acute_foodweb_edgelist.csv', sep='\t')
print(acute_edgelist.shape)
acute_edgelist.head()

In [ ]:
acute_chem_int = set(acute_edgelist['Source']).intersection(set(acute_ssd_hc5.keys()))
len(acute_chem_int)

In [ ]:
acute_prey_int = set(acute_edgelist[acute_edgelist['Source'].isin(acute_chem_int)]['Target'])
len(acute_prey_int)

In [ ]:
acute_edgelist = acute_edgelist[acute_edgelist['Source'].isin(acute_chem_int.union(acute_prey_int))]
print(acute_edgelist.shape)
acute_edgelist.head()

In [ ]:
# Check
act_prey = set(acute_edgelist['Source']).intersection(acute_prey_int)
act_pred = set(acute_edgelist['Target']) - act_prey

print(len(set(set(acute_edgelist['Target']))))
print(len(act_prey), len(act_pred), len(act_prey.union(act_pred)))

print(len(set(set(acute_edgelist['Source']))))
print(len(acute_chem_int), len(act_prey))

In [ ]:
acute_edgelist['Weight'] = acute_edgelist['Weight'].astype(float)
acute_edgelist = acute_edgelist.groupby(by=['Source', 'Target'])['Weight'].agg(lambda x: gmean(x[x > 0]) if (x > 0).any() else 0).reset_index()
print(acute_edgelist.shape)
acute_edgelist.head()

In [ ]:
def acute_hc5_comparison(row):
    id = row['Source']
    wgt = row['Weight']
    if id in acute_chem_int:
        if float(wgt) < float(acute_ssd_hc5[id]):
            return 'Sensitive'
        else:
            return 'Not Sensitive'
    else:
        return '' 

In [ ]:
acute_edgelist['Type'] = acute_edgelist.apply(acute_hc5_comparison, axis=1)
print(acute_edgelist.shape)
acute_edgelist.head()

In [ ]:
acute_edgelist[acute_edgelist['Weight']>0]

In [ ]:
acute_edgelist['Type'].value_counts()

In [ ]:
acute_nodelist = pd.read_csv('./acute_foodweb_nodelist.csv')
print(acute_nodelist.shape)
acute_nodelist = acute_nodelist.sort_values(by='node_type', ascending=False)
acute_nodelist.head()

In [ ]:
print(len(set(acute_edgelist['Source']).union(set(acute_edgelist['Target']))))

In [ ]:
acute_nodelist = acute_nodelist[acute_nodelist['node'].isin(set(acute_edgelist['Source']).union(set(acute_edgelist['Target'])))]
print(acute_nodelist.shape)
acute_nodelist.head()

In [ ]:
acute_nodelist['node_type'].value_counts()

In [ ]:
acute_edgelist.to_csv('./filtered_acute_foodweb_edgelist.tsv', sep='\t', index=False, encoding='utf-8')
acute_nodelist.to_csv('./filtered_acute_foodweb_nodelist.tsv', sep='\t', index=False, encoding='utf-8')

In [ ]:
G_acute = nx.from_pandas_edgelist(acute_edgelist, 'Source', 'Target', 
                              edge_attr=['Weight', 'Type'], create_using=nx.DiGraph())

In [ ]:
print(f"Nodes in largest WCC: {G_acute.number_of_nodes()}")
print(f"Edges in largest WCC: {G_acute.number_of_edges()}")

In [ ]:
# Extract largest WCC as a directed subgraph
largest_wcc = max(nx.weakly_connected_components(G_acute), key=len)
wcc_nodes = largest_wcc
G = G_acute.subgraph(wcc_nodes).copy()

print(f"Nodes in largest WCC: {G.number_of_nodes()}")
print(f"Edges in largest WCC: {G.number_of_edges()}")

in_deg = G.in_degree()
out_deg = G.out_degree()

top_products = sorted(in_deg, key=lambda x: x[1], reverse=True)[:10]
top_reactants = sorted(out_deg, key=lambda x: x[1], reverse=True)[:10]

print("Highly produced species:", top_products)
print("Highly transforming species:", top_reactants)

In [ ]:
res = analyze_chemical_predator_paths(G_acute, acute_nodelist)

In [ ]:
aquaculture_chem[aquaculture_chem['id'].isin(['RID00113', 'RID00053', 'RID00092'])]

# Chronic toxicity SSD

### Chronic HC05 values

In [ ]:
chronic_ssd = pd.read_csv('./external_data/chronic_averaged_hc05_all_chemicals.tsv', sep='\t')
print(chronic_ssd.shape)
chronic_ssd.head()

In [ ]:
print(len(set(chronic_ssd['Chemical'])))
print(len(set(chronic_ssd['is_bestfit'])))

In [ ]:
chronic_ssd['id'] = chronic_ssd['cas'].apply(lambda x: chemdict[x] if x in set(chemdict.keys()) else x)
print(chronic_ssd.shape)
chronic_ssd.head()

In [ ]:
chronic_ssd_hc5 = dict(zip(chronic_ssd['id'], chronic_ssd['est']))
len(chronic_ssd_hc5)

### chronic trophic network

In [ ]:
chronic_edgelist = pd.read_csv('./chronic_foodweb_edgelist.csv', sep='\t')
print(chronic_edgelist.shape)
chronic_edgelist.head()

In [ ]:
chronic_chem_int = set(chronic_edgelist['Source']).intersection(set(chronic_ssd_hc5.keys()))
len(chronic_chem_int)

In [ ]:
chronic_prey_int = set(chronic_edgelist[chronic_edgelist['Source'].isin(chronic_chem_int)]['Target'])
len(chronic_prey_int)

In [ ]:
chronic_edgelist = chronic_edgelist[chronic_edgelist['Source'].isin(chronic_chem_int.union(chronic_prey_int))]
print(chronic_edgelist.shape)
chronic_edgelist.head()

In [ ]:
# Check
chr_prey = set(chronic_edgelist['Source']).intersection(chronic_prey_int)
chr_pred = set(chronic_edgelist['Target']) - chr_prey

print(len(set(set(chronic_edgelist['Target']))))
print(len(chr_prey), len(chr_pred), len(chr_prey.union(chr_pred)))

print(len(set(set(chronic_edgelist['Source']))))
print(len(chronic_chem_int), len(chr_prey))

In [ ]:
chronic_edgelist['Weight'] = chronic_edgelist['Weight'].astype(float)
chronic_edgelist = chronic_edgelist.groupby(by=['Source', 'Target'])['Weight'].agg(lambda x: gmean(x[x > 0]) if (x > 0).any() else 0).reset_index()
print(chronic_edgelist.shape)
chronic_edgelist.head()

In [ ]:
def chronic_hc5_comparison(row):
    id = row['Source']
    wgt = row['Weight']
    if id in chronic_chem_int:
        if float(wgt) < float(chronic_ssd_hc5[id]):
            return 'Sensitive'
        else:
            return 'Not Sensitive'
    else:
        return '' 

In [ ]:
chronic_edgelist['Type'] = chronic_edgelist.apply(chronic_hc5_comparison, axis=1)
print(chronic_edgelist.shape)
chronic_edgelist.head()

In [ ]:
chronic_edgelist['Type'].value_counts()

In [ ]:
chronic_nodelist = pd.read_csv('./chronic_foodweb_nodelist.csv')
print(chronic_nodelist.shape)
chronic_nodelist = chronic_nodelist.sort_values(by='node_type', ascending=False)
chronic_nodelist.head()

In [ ]:
print(len(set(chronic_edgelist['Source']).union(set(chronic_edgelist['Target']))))

In [ ]:
chronic_nodelist = chronic_nodelist[chronic_nodelist['node'].isin(set(chronic_edgelist['Source']).union(set(chronic_edgelist['Target'])))]
print(chronic_nodelist.shape)
chronic_nodelist.head()

In [ ]:
chronic_nodelist['node_type'].value_counts()

In [ ]:
chronic_edgelist.to_csv('./filtered_chronic_foodweb_edgelist.tsv', sep='\t', index=False, encoding='utf-8')
chronic_nodelist.to_csv('./filtered_chronic_foodweb_nodelist.tsv', sep='\t', index=False, encoding='utf-8')

In [ ]:
G_chronic = nx.from_pandas_edgelist(chronic_edgelist, 'Source', 'Target', 
                              edge_attr=['Weight', 'Type'], create_using=nx.DiGraph())

In [ ]:
# Extract largest WCC as a directed subgraph
largest_wcc = max(nx.weakly_connected_components(G_chronic), key=len)
wcc_nodes = largest_wcc
G = G_chronic.subgraph(wcc_nodes).copy()

print(f"Nodes in largest WCC: {G.number_of_nodes()}")
print(f"Edges in largest WCC: {G.number_of_edges()}")

in_deg = G.in_degree()
out_deg = G.out_degree()

top_products = sorted(in_deg, key=lambda x: x[1], reverse=True)[:10]
top_reactants = sorted(out_deg, key=lambda x: x[1], reverse=True)[:10]

print("Highly produced species:", top_products)
print("Highly transforming species:", top_reactants)

In [ ]:
res = analyze_chemical_predator_paths(G_chronic, chronic_nodelist)

In [ ]:
aquaculture_chem[aquaculture_chem['id'].isin(['RID00080', 'RID00105', 'RID00137', 'RID00185', 'RID00252'])]

## Betweenness centrality

Acute

In [ ]:
prey_acute_only = set(acute_nodelist[acute_nodelist['node_type'].isin(['prey'])]['node'])
prey_acute = set(acute_nodelist[acute_nodelist['node_type'].isin(['prey', 'both'])]['node'])

bet_centrality_acute = nx.betweenness_centrality(G_acute)

bet_centrality_acute_prey_only = ((k,v) for k,v in bet_centrality_acute.items() if k in prey_acute_only)
bet_centrality_acute_prey_only_sorted = sorted(bet_centrality_acute_prey_only, key=lambda x:x[1], reverse=True)
bet_centrality_acute_prey_only_sorted

In [ ]:
# Initialize a new column with NaN
acute_nodelist['betweenness_centrality'] = np.nan
acute_nodelist['betweenness_centrality'] = acute_nodelist['node'].map(bet_centrality_acute)

# Optional: check the updated table
acute_nodelist.head()
acute_nodelist.to_csv('./acute_filtered_foodweb_nodelist_betweenness.tsv', sep='\t', index=None, encoding='utf-8')

Chronic

In [ ]:
prey_chronic_only = set(chronic_nodelist[chronic_nodelist['node_type'].isin(['prey'])]['node'])
prey_chronic = set(chronic_nodelist[chronic_nodelist['node_type'].isin(['prey', 'both'])]['node'])

bet_centrality_chronic = nx.betweenness_centrality(G_chronic)

bet_centrality_chronic_prey_only = ((k,v) for k,v in bet_centrality_chronic.items() if k in prey_chronic_only)
bet_centrality_chronic_prey_only_sorted = sorted(bet_centrality_chronic_prey_only, key=lambda x:x[1], reverse=True)
bet_centrality_chronic_prey_only_sorted

In [ ]:
# Initialize a new column with NaN
chronic_nodelist['betweenness_centrality'] = np.nan
chronic_nodelist['betweenness_centrality'] = chronic_nodelist['node'].map(bet_centrality_chronic)

# Optional: check the updated table
chronic_nodelist.head()
chronic_nodelist.to_csv('./chronic_filtered_foodweb_nodelist_betweenness.tsv', sep='\t', index=None, encoding='utf-8')